# 03 – Synthetic candidates and ground truth

**Step 1 (candidates) and the basis of Step 3 (evaluation).**

Real, consented job-seeker data is not available, so we generate **synthetic Limpopo job seekers** whose correct matches are known in advance. This notebook:

1. Groups the 427 jobs into **job families** (Finance, Nursing, Mining, Trades…) using editable title rules.
2. Extracts each job's **mandatory credentials** (ignoring anything marked "advantage", "preferred" or "desirable"; treating "A **or** B" as alternatives).
3. Generates candidates from a **seed job**: ESCO skills for the family's occupations, credentials based on the seed job, a home town, mobility, group attributes, and training constraints.
4. Builds the **ground truth**: how relevant every job in the family is to each candidate.

**Three candidate scenarios** (shares set in the configuration cell):
- **exact**: holds the seed job's credentials in the standard wording;
- **equivalent**: holds credentials that meet the requirements under a *verified* equivalence rule, in local wording (e.g. *NC(V) Level 4* instead of *Grade 12*, *Code 10* instead of *Code 08*);
- **under**: lacks or holds a lower version of one requirement.

**Relevance (ground truth)**, defined independently of either matcher and using only *verified* equivalence rules:

| rel | Meaning |
|---|---|
| 2 | Same job family **and** formally eligible **and** within the candidate's reachable districts |
| 1 | Same job family, but not eligible or not reachable |
| 0 | Different job family |

`rel_occupation` (1 = same family) is also saved, so results can be checked without the eligibility and location assumptions.

In [16]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.limpopo import (read_csv_any_encoding, LocalVocab, parse_requirements, requirements_text,
                         is_eligible, assign_family, reachable_districts, DISTRICTS, HOME_TOWNS, ADJACENT, EDU_CATS)

RAW, LOCAL, PROCESSED = ROOT / "data" / "raw", ROOT / "data" / "raw" / "local", ROOT / "data" / "processed"

jobs = pd.read_csv(PROCESSED / "jobs_clean.csv", dtype=str).fillna("")
lv = LocalVocab(read_csv_any_encoding(LOCAL / "local_vocabulary.csv"), read_csv_any_encoding(LOCAL / "equivalences.csv"))
rules = read_csv_any_encoding(LOCAL / "job_family_rules.csv").sort_values("priority", key=lambda s: s.astype(int))
families = read_csv_any_encoding(LOCAL / "job_families.csv").set_index("family")
print(f"{len(jobs)} jobs, {len(lv.vocab)} local vocabulary entries, {len(rules)} family rules, {len(families)} families")

427 jobs, 85 local vocabulary entries, 29 family rules, 23 families


## 1. Job families
The first matching rule in `job_family_rules.csv` decides a job's family. Check the sample below; fix any wrong one by adding a rule near the top of the file.

In [17]:
jobs["family"] = jobs["title"].map(lambda t: assign_family(t, rules))
print("Unassigned jobs:", (jobs["family"] == "UNASSIGNED").sum())
fam_counts = jobs["family"].value_counts().rename("jobs").to_frame()
fam_counts["description"] = fam_counts.index.map(lambda f: families["description"].get(f, ""))
fam_counts

Unassigned jobs: 0


,jobs,description
family,,
ENGINEERING,50,"Engineers, quantity surveyors, architects, con..."
FINANCE,46,"Accounting, budgeting, payroll, audit, risk an..."
HR,45,"HR administration, recruitment, training, labo..."
SCM,29,"Procurement, supply chain, stores, assets and ..."
SOCIAL,26,"Social workers, community development, child a..."
EDUCATION,23,"Education management, circuit management, lect..."
ADMIN,23,"Clerical, secretarial, records and office admi..."
TRANSPORT_REG,22,"Traffic law enforcement, transport regulation,..."
HEALTH,21,"Doctors, pharmacists, therapists, radiographer..."


In [18]:
# Spot-check: two random titles per family
jobs.groupby("family").sample(n=2, random_state=1, replace=True)[["family", "title"]].drop_duplicates().sort_values("family")

,family,title
137,ADMIN,Personal Assistant (Corporate Services)
207,ADMIN,Administration Clerk: Lease and Municipal Serv...
364,AGRICULTURE,Technical Sales Representative (Agriculture)
102,EDUCATION,Senior Administrative Officer: Examinations an...
291,ELEMENTARY,Cleaner
21,ELEMENTARY,General Worker
336,EMS,EMS Station Manager Grade 3 (Shifts)
228,ENGINEERING,Quantity Surveyor (Grade A-C)
218,ENGINEERING,"Chief Construction Project Manager, Grade A"
142,ENVIRONMENT,Field Ranger (Wildlife Resources)


## 2. Mandatory credential requirements
Each requirement is an *OR-group*: the candidate needs at least one item from every group. Example: `[['GR12','N3'], ['TRADE_TEST'], ['LIC_ANY']]` means *(Grade 12 or N3) and a trade test and a driver's licence*.

In [19]:
jobs["req_groups"] = jobs["description"].map(lambda d: parse_requirements(d, lv))
show = jobs.sample(6, random_state=3)
for _, r in show.iterrows():
    print(f"{r['title']}\n   Advert: {requirements_text(r['description'])[:160].strip()}…\n   Groups: {r['req_groups']}\n")
print(f"Jobs with at least one mandatory credential: {(jobs['req_groups'].map(len) > 0).mean():.0%}")
print(f"Average number of requirement groups per job: {jobs['req_groups'].map(len).mean():.1f}")

Paramedic (ALS/ECP) - Emergency Services
   Advert: Advanced Life Support qualification (CCA or NDip) or equivalent (essential); HPCSA registration as ALS Paramedic or Emergency Care Practitioner; valid driver's…
   Groups: [['DIPLOMA'], ['EMS_QUAL'], ['REG_HPCSA'], ['LIC_ANY'], ['PRDP'], ['EMS_CPG']]

Chief Engineer: Mechanical: Engineering Services, Grade A-B
   Advert: Mechanical engineering degree; ECSA registration as Professional Engineer; six years' experience; health-care experience an advantage; willingness to travel.…
   Groups: [['DEGREE'], ['REG_ECSA']]

Graduate Internship Programme 2027/2029: Properties & Facilities
   Advert: N Dip/Degree in Real Estate or Property Science; or N Dip/Degree in Horticulture.…
   Groups: [['DEGREE', 'DIPLOMA']]

Ambulance Emergency Assistant (AEA/ILS) - Emergency Services
   Advert: AEA/ILS qualification and HPCSA registration as AEA/ILS (essential); valid driver's licence with PrDP; prior EMS experience (essential); 1 year AEA/ILS operati…

## 3. ESCO occupations for each family
Candidates take their skills from ESCO occupations whose ISCO code starts with the family's `isco_prefixes`.

In [20]:
occ = pd.read_csv(PROCESSED / "esco_occupations.csv", dtype=str).fillna("")
rel = pd.read_csv(PROCESSED / "esco_occupation_skills.csv", dtype=str).fillna("")
skills = pd.read_csv(RAW / "esco" / "skills_en.csv", dtype=str).fillna("")
SKILL_LABEL = dict(zip(skills["conceptUri"], skills["preferredLabel"]))
OCC_SKILLS = {k: g for k, g in rel.groupby("occupationUri")}

def family_occupations(fam):
    prefixes = tuple(p for p in families.loc[fam, "isco_prefixes"].split(";") if p)
    o = occ[occ["iscoGroup"].astype(str).str.startswith(prefixes) & occ["conceptUri"].isin(OCC_SKILLS.keys())]
    return o

FAM_OCC = {f: family_occupations(f) for f in fam_counts.index}
pd.DataFrame({"jobs": fam_counts["jobs"], "esco_occupations": {f: len(o) for f, o in FAM_OCC.items()}})

,jobs,esco_occupations
ENGINEERING,50,265
FINANCE,46,59
HR,45,16
SCM,29,147
SOCIAL,26,61
EDUCATION,23,118
ADMIN,23,29
TRANSPORT_REG,22,30
HEALTH,21,48
ICT,19,97


## 4. Configuration
Change these and re-run to test how sensitive your results are (e.g. a larger *equivalent* share).

In [21]:
CONFIG = {
    "N_CANDIDATES": 600,
    "SEED": 42,
    "SCENARIO_SHARES": {"exact": 0.40, "equivalent": 0.30, "under": 0.30},
    "MOBILITY_SHARES": {"local": 0.50, "regional": 0.35, "province": 0.15},
    "P_HOME_IN_SEED_DISTRICT": 0.60,     # otherwise a random other district
    "MIN_JOBS_PER_FAMILY": 3,            # families with fewer jobs are not used as seeds
    "SKILLS_ESSENTIAL_SHARE": (0.5, 0.9),
    "N_NOISE_SKILLS": 2,
}
GENDERS = (["Female", "Male"], [0.5, 0.5])
AGE_BANDS = (["18-24", "25-34", "35-44", "45-59"], [0.30, 0.35, 0.20, 0.15])
EXPERIENCE = {"18-24": (0, 3), "25-34": (1, 8), "35-44": (4, 15), "45-59": (8, 25)}
TRAIN_HOURS = ([5, 10, 20], [0.4, 0.4, 0.2])
TRAIN_BUDGET = ([0, 500, 2000, 5000, 15000], [0.30, 0.25, 0.20, 0.15, 0.10])

# Equivalent credentials (only kept if a VERIFIED rule makes them satisfy the requirement)
ALTERNATIVES = {"GR12": ["NCV_L4"], "BASIC_ED": ["NCV_L4", "GR12"], "LIC_ANY": ["LIC_C1", "LIC_EB", "LIC_EC"],
                "LIC_B": ["LIC_C1", "LIC_EB"], "LIC_C1": ["LIC_C", "LIC_EC"], "LIC_EB": ["LIC_EC"],
                "N4": ["N5", "N6"], "N5": ["N6"], "N6": ["NDIP_N"], "NCV_L3": ["NCV_L4"],
                "HIGHER_CERT": ["DIPLOMA"], "DIPLOMA": ["DEGREE"], "DEGREE": ["HONOURS"]}
ALTERNATIVES = {req: [a for a in alts if lv.satisfies({a}, req)] for req, alts in ALTERNATIVES.items()}
LOWER = {"HONOURS": "DEGREE", "DEGREE": "DIPLOMA", "DIPLOMA": "GR12", "HIGHER_CERT": "GR12", "NDIP_N": "N6",
         "N6": "N4", "N5": "N4", "N4": "N3", "NCV_L4": "NCV_L3", "GR12": "BASIC_ED"}
EDU_RANK = {"BASIC_ED": 0, "NCV_L2": 0, "NCV_L3": 0, "N1": 0, "N2": 0, "GR12": 1, "NCV_L4": 1, "N3": 1,
            "N4": 2, "N5": 2, "N6": 2, "NDIP_N": 2, "TRADE_TEST": 2, "HIGHER_CERT": 3, "DIPLOMA": 3,
            "DEGREE": 4, "HONOURS": 4}
DISPLAY = {"DEGREE": ["Bachelor's degree", "BTech", "Degree"], "DIPLOMA": ["National Diploma", "Diploma"],
           "HONOURS": ["Honours degree", "Postgraduate diploma"], "HIGHER_CERT": ["Higher Certificate"],
           "NDIP_N": ["National N Diploma", "N Diploma"]}
EDU_LEVEL_NAME = {0: "Below matric", 1: "Matric / NC(V) L4", 2: "TVET or trade", 3: "Diploma", 4: "Degree or higher"}
print("Verified alternatives:", {k: v for k, v in ALTERNATIVES.items() if v})

Verified alternatives: {'GR12': ['NCV_L4'], 'BASIC_ED': ['NCV_L4', 'GR12'], 'LIC_ANY': ['LIC_C1', 'LIC_EB', 'LIC_EC'], 'LIC_B': ['LIC_C1', 'LIC_EB'], 'LIC_C1': ['LIC_C', 'LIC_EC'], 'LIC_EB': ['LIC_EC'], 'N4': ['N5', 'N6'], 'N5': ['N6'], 'N6': ['NDIP_N'], 'NCV_L3': ['NCV_L4'], 'HIGHER_CERT': ['DIPLOMA'], 'DIPLOMA': ['DEGREE'], 'DEGREE': ['HONOURS']}


## 5. Candidate generator

In [22]:
rng = np.random.default_rng(CONFIG["SEED"])
pick = lambda opts: opts[0][rng.choice(len(opts[0]), p=opts[1])]

def cred_text(cid, field, standard):
    cat = lv.category[cid]
    if cid == "LIC_ANY":
        return "Valid driver's licence"
    aliases = lv.aliases.get(cid) or []
    if cid in DISPLAY:                                   # sensible wording for qualifications
        aliases = DISPLAY[cid]
    if cat in ("registration", "mining", "safety", "health", "public_service", "occupational") or not aliases:
        name = lv.label[cid]
    else:
        name = aliases[0] if standard else aliases[rng.integers(len(aliases))]
    if cat in ("higher", "tvet") and field and cid not in ("N1", "N2", "N3", "N4", "N5", "N6"):
        name = f"{name} in {field}"
    elif cid in ("LIC_B", "LIC_EB", "LIC_C1", "LIC_C", "LIC_EC"):
        name = f"{name} driver's licence"
    return name

def build_credentials(groups, scenario):
    """Return (set of credential ids, scenario actually applied)."""
    held = [g[0] for g in groups]                       # exact: first option of each group
    applied = scenario
    if scenario == "equivalent":
        swappable = [i for i, c in enumerate(held) if ALTERNATIVES.get(c)]
        if swappable:
            for i in swappable:
                held[i] = rng.choice(ALTERNATIVES[held[i]])
        else:
            applied = "exact"
    elif scenario == "under":
        if held:
            i = rng.integers(len(held))
            if held[i] in LOWER:
                held[i] = LOWER[held[i]]
            else:
                held.pop(i)
        else:
            applied = "exact"
    creds = set(held)
    if not any(lv.category.get(c) in EDU_CATS for c in creds):           # everyone has some schooling
        creds.add("GR12" if rng.random() < 0.7 else "BASIC_ED")
    return creds, applied

def sample_skills(fam):
    occs = FAM_OCC[fam]
    o = occs.iloc[rng.integers(len(occs))]
    r = OCC_SKILLS[o["conceptUri"]]
    ess = r[r["relationType"] == "essential"]["skillUri"].tolist()
    opt = r[r["relationType"] != "essential"]["skillUri"].tolist()
    lo, hi = CONFIG["SKILLS_ESSENTIAL_SHARE"]
    k = max(1, int(round(len(ess) * rng.uniform(lo, hi)))) if ess else 0
    chosen = list(rng.choice(ess, size=k, replace=False)) if k else []
    if opt:
        chosen += list(rng.choice(opt, size=min(len(opt), int(rng.integers(0, 4))), replace=False))
    noise_pool = rel["skillUri"].unique()
    chosen += list(rng.choice(noise_pool, size=CONFIG["N_NOISE_SKILLS"], replace=False))
    return o, list(dict.fromkeys(chosen))

def edu_level(creds):
    ranks = [EDU_RANK[c] for c in creds if c in EDU_RANK]
    return EDU_LEVEL_NAME[max(ranks) if ranks else 0]

seed_families = [f for f, n in fam_counts["jobs"].items() if n >= CONFIG["MIN_JOBS_PER_FAMILY"] and len(FAM_OCC[f]) > 0]
print("Families used as seeds:", len(seed_families), "| skipped:", sorted(set(fam_counts.index) - set(seed_families)))

Families used as seeds: 23 | skipped: []


In [23]:
scen_names, scen_p = list(CONFIG["SCENARIO_SHARES"]), list(CONFIG["SCENARIO_SHARES"].values())
mob_names, mob_p = list(CONFIG["MOBILITY_SHARES"]), list(CONFIG["MOBILITY_SHARES"].values())
MOBILITY_TEXT = {"local": "in {d} District only", "regional": "in {d} District and neighbouring districts",
                 "province": "anywhere in Limpopo"}

rows = []
for n in range(CONFIG["N_CANDIDATES"]):
    fam = seed_families[rng.integers(len(seed_families))]              # families equally likely
    seed = jobs[jobs["family"] == fam].sample(1, random_state=int(rng.integers(1e9))).iloc[0]
    scenario = rng.choice(scen_names, p=scen_p)
    creds, applied = build_credentials(seed["req_groups"], scenario)
    seed_d = seed["district"] if seed["district"] in DISTRICTS else rng.choice(DISTRICTS)
    home_d = seed_d if rng.random() < CONFIG["P_HOME_IN_SEED_DISTRICT"] else rng.choice([d for d in DISTRICTS if d != seed_d])
    town = rng.choice(HOME_TOWNS[home_d])
    mobility = rng.choice(mob_names, p=mob_p)
    age = pick(AGE_BANDS); lo, hi = EXPERIENCE[age]
    years = int(rng.integers(lo, hi + 1))
    o, skill_uris = sample_skills(fam)
    field_opts = [f for f in families.loc[fam, "study_fields"].split(";") if f]
    field = rng.choice(field_opts) if field_opts else ""
    standard = applied != "equivalent"
    cred_txt = [cred_text(c, field, standard) for c in sorted(creds, key=lambda c: -EDU_RANK.get(c, -1))]
    skill_txt = [SKILL_LABEL.get(u, "") for u in skill_uris][:12]
    profile = (f"Job seeker from {town}, {home_d} District. "
               f"Qualifications: {'; '.join(cred_txt)}. "
               f"Experience: {years} year{'' if years == 1 else 's'} as {o['preferredLabel']}. "
               f"Skills: {', '.join(s for s in skill_txt if s)}. "
               f"Willing to work {MOBILITY_TEXT[mobility].format(d=home_d)}.")
    rows.append({
        "candidate_id": f"C{n+1:04d}", "family": fam, "seed_job_id": seed["job_id"],
        "scenario": applied, "esco_occupation_uri": o["conceptUri"], "occupation_label": o["preferredLabel"],
        "home_town": town, "home_district": home_d, "mobility": mobility,
        "reachable_districts": ";".join(sorted(reachable_districts(home_d, mobility))),
        "gender": pick(GENDERS), "age_band": age, "education_level": edu_level(creds),
        "experience_years": years, "train_hours_week": int(pick(TRAIN_HOURS)), "train_budget_rand": int(pick(TRAIN_BUDGET)),
        "credentials": ";".join(sorted(creds)), "skill_uris": ";".join(skill_uris), "profile_text": profile,
    })
cands = pd.DataFrame(rows)
print(f"Generated {len(cands)} candidates")
for t in cands.sample(4, random_state=7)["profile_text"]:
    print("\n" + t)

Generated 600 candidates

Job seeker from Marble Hall, Sekhukhune District. Qualifications: Honours degree in Education Management; SMS Pre-Entry (Nyukela) certificate; Code EC driver's licence. Experience: 14 years as prison instructor. Skills: perform classroom management, comply with the principles of self-defence, apply knowledge of human behaviour, practice vigilance, assess offenders' risk behaviour, apply teaching strategies, correctional procedures, apply intercultural teaching strategies, provide lesson materials, adult education, contribute to the formulation of correctional procedures, conduct frisk. Willing to work in Sekhukhune District only.

Job seeker from Seshego, Capricorn District. Qualifications: Grade 12. Experience: 1 year as hospitality establishment receptionist. Skills: comply with food safety and hygiene, handover the service area, maintain customer service, handle financial transactions, assist clients with special needs, assist at check-in, implement sales s

## 6. Ground truth

In [24]:
gt_rows = []
for _, c in cands.iterrows():
    creds = set(c["credentials"].split(";"))
    reach = set(c["reachable_districts"].split(";"))
    fam_jobs = jobs[jobs["family"] == c["family"]]
    for _, j in fam_jobs.iterrows():
        eligible = is_eligible(creds, j["req_groups"], lv, verified_only=True)
        reachable = j["district"] in reach
        gt_rows.append({"candidate_id": c["candidate_id"], "job_id": j["job_id"],
                        "rel": 2 if (eligible and reachable) else 1, "rel_occupation": 1,
                        "eligible": eligible, "reachable": reachable,
                        "is_seed": j["job_id"] == c["seed_job_id"]})
gt = pd.DataFrame(gt_rows)          # jobs not listed for a candidate have rel = 0

per_cand = gt.groupby("candidate_id").agg(rel2=("rel", lambda s: (s == 2).sum()), family_jobs=("rel", "size"))
per_cand = per_cand.join(cands.set_index("candidate_id")[["scenario", "mobility"]])
print(f"Average jobs in the candidate's family: {per_cand['family_jobs'].mean():.1f}")
print(f"Average fully relevant jobs (rel = 2): {per_cand['rel2'].mean():.1f}")
print(f"Candidates with at least one rel = 2 job: {(per_cand['rel2'] > 0).mean():.0%}\n")
per_cand.groupby("scenario")["rel2"].agg(candidates="size", mean_rel2_jobs="mean",
                                         share_with_any=lambda s: (s > 0).mean()).round(2)

Average jobs in the candidate's family: 18.9
Average fully relevant jobs (rel = 2): 6.1
Candidates with at least one rel = 2 job: 80%



,candidates,mean_rel2_jobs,share_with_any
scenario,,,
equivalent,128,7.24,0.89
exact,297,7.62,0.91
under,175,2.65,0.54


In [25]:
# Sanity check: is each candidate eligible for their own seed job?
seed = gt[gt["is_seed"]].merge(cands[["candidate_id", "scenario"]], on="candidate_id")
seed.groupby("scenario")[["eligible", "reachable"]].mean().round(2)

,eligible,reachable
scenario,,
equivalent,1.00,0.66
exact,1.00,0.77
under,0.24,0.75


Expected: *exact* and *equivalent* candidates are (almost) all eligible for their seed job; *under* candidates mostly are not. If not, check the requirement groups of those seed jobs.

## 7. Candidate attributes (for the fairness analysis)

In [26]:
for col in ["scenario", "gender", "age_band", "education_level", "home_district", "mobility"]:
    print(cands[col].value_counts(normalize=True).round(2).to_string(), "\n")

scenario
exact         0.50
under         0.29
equivalent    0.21 

gender
Female    0.5
Male      0.5 

age_band
25-34    0.37
18-24    0.29
35-44    0.19
45-59    0.16 

education_level
Matric / NC(V) L4    0.31
Diploma              0.30
Degree or higher     0.26
Below matric         0.10
TVET or trade        0.03 

home_district
Capricorn     0.38
Waterberg     0.17
Sekhukhune    0.17
Mopani        0.15
Vhembe        0.14 

mobility
local       0.51
regional    0.37
province    0.12 



## 8. Save

In [27]:
cands.to_csv(PROCESSED / "candidates.csv", index=False)
gt.to_csv(PROCESSED / "ground_truth.csv", index=False)
jobs_out = jobs.assign(req_groups=jobs["req_groups"].map(json.dumps))
jobs_out[["job_id", "title", "family", "district", "town", "source", "req_groups"]].to_csv(PROCESSED / "job_requirements.csv", index=False)
json.dump(CONFIG, open(PROCESSED / "candidates_config.json", "w"), indent=2)
for f in ["candidates.csv", "ground_truth.csv", "job_requirements.csv", "candidates_config.json"]:
    print(f"{f:28s} {(PROCESSED / f).stat().st_size/1024:8.0f} KB")

candidates.csv                   1352 KB
ground_truth.csv                  420 KB
job_requirements.csv               53 KB
candidates_config.json              0 KB


### Notes for the write-up
- **Synthetic candidates:** no real job seekers were used. Profiles are generated from ESCO occupations and the Limpopo adverts; results show how the matchers behave under controlled conditions, not real hiring outcomes.
- **Ground truth assumptions:** a good match is defined as same job family + formally eligible + reachable. Local-aware mode is designed around the same ideas, so also report results with `rel_occupation` (family only).
- **Verified rules only:** eligibility in the ground truth uses only equivalence rules marked `verified = 1`.
- **Job families** are researcher-defined from job titles (23 families); seniority and years of experience are not modelled.
- **Scenario shares** (40% exact / 30% equivalent / 30% under) are a design choice; report a sensitivity run with different shares.
- **District adjacency** (`ADJACENT` in `src/limpopo.py`) is an assumption based on the district map.
- **Group attributes** (gender, age) are assigned at random, independent of qualifications, so any difference the matchers show between these groups comes from proxies such as district or education level.